# Laboratorio 03 — CDC, SCD1 y SCD2 con Lakeflow

**Semana:** 05 | **Actividad de referencia:** Actividad 03  
**Modalidad:** Individual | **Entorno:** Databricks Lakeflow (Spark Declarative Pipelines)

---

## Instrucciones generales

Simula un proceso de **Change Data Capture (CDC)** sobre tu dataset propio usando `dp.create_auto_cdc_flow()` y `dp.create_streaming_table()`. Implementa tanto **SCD Tipo 1** (sobreescribir) como **SCD Tipo 2** (historial completo de cambios).

> **Importante:** Este notebook debe ejecutarse como pipeline Lakeflow. Las celdas de código definen las transformaciones declarativas del grafo.

## Parte 1 — Descripción del dataset y diseño CDC

1. **Nombre, fuente y URL** del dataset.
2. **Clave primaria:** ¿Qué columna(s) identifican unívocamente cada entidad? (ej: `user_id`, `product_id`).
3. **Columna de secuencia:** ¿Qué columna determina el orden de los cambios? (ej: `updated_at`, `version`, `timestamp`).
4. **Tipo de operaciones CDC simuladas:** INSERT, UPDATE, DELETE.
5. **¿Por qué SCD1 o SCD2?** Para tu dataset, ¿es más útil conservar el historial (SCD2) o solo el último estado (SCD1)?

**Escribe tu respuesta aquí:**

## Parte 2 — Importaciones y parámetros de pipeline

In [ ]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F

RUTA_CDC        = spark.conf.get("ruta_cdc",    "/Volumes/workspace/default/week_5/cdc/")
SCHEMA_LOCATION = spark.conf.get("schema_loc",  "/Volumes/workspace/default/week_5/schema_cdc/")
ENTORNO         = spark.conf.get("entorno",      "dev")

# Nombre de la columna clave primaria en tu dataset — ajusta
CLAVE_PK        = "columna_id"      # reemplaza con el nombre real
COL_SECUENCIA   = "columna_fecha"   # reemplaza con el nombre real (timestamp o version)
COL_OPERACION   = "operacion"       # columna con valor 'INSERT', 'UPDATE', 'DELETE'

print(f"ruta_cdc       = {RUTA_CDC}")
print(f"clave_pk       = {CLAVE_PK}")
print(f"col_secuencia  = {COL_SECUENCIA}")

## Parte 3 — Perfil técnico y simulación de cambios CDC (exploración interactiva)

In [ ]:
# Leer el dataset original para entender su estructura
df_original = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load(f"{RUTA_CDC}/*.csv")

print(f"Registros originales: {df_original.count():,}")
df_original.printSchema()
df_original.show(5, truncate=False)

In [ ]:
# Simular eventos CDC si tu dataset no incluye columna 'operacion'
# Crea un DataFrame simulado con operaciones INSERT, UPDATE, DELETE
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
from datetime import datetime

# Ajusta este bloque a los tipos reales de tu dataset
data_cdc_simulada = [
    # (clave, campo_a_cambiar, timestamp, operacion)
    ("ID001", "valor_original", datetime(2024, 1, 1), "INSERT"),
    ("ID002", "valor_inicial",  datetime(2024, 1, 2), "INSERT"),
    ("ID001", "valor_actualizado", datetime(2024, 2, 1), "UPDATE"),  # actualiza ID001
    ("ID003", "nuevo_valor",    datetime(2024, 3, 1), "INSERT"),
    ("ID002", None,             datetime(2024, 4, 1), "DELETE"),     # elimina ID002
]

schema_cdc = StructType([
    StructField("columna_id",        StringType(),    False),
    StructField("columna_valor",     StringType(),    True),
    StructField("columna_fecha",     TimestampType(), False),
    StructField("operacion",         StringType(),    False)
])

df_cdc_sim = spark.createDataFrame(data_cdc_simulada, schema_cdc)
df_cdc_sim.show(truncate=False)

# Guardar en el directorio CDC para que Auto Loader lo lea
df_cdc_sim.coalesce(1).write.mode("overwrite").format("csv") \
    .option("header", True) \
    .save(f"{RUTA_CDC}")
print(f"✓ Datos CDC simulados guardados en {RUTA_CDC}")

**Análisis del CDC simulado:**  
¿Cuántos INSERTs, UPDATEs y DELETEs hay? ¿El orden de los timestamps es coherente? ¿Hay alguna clave que aparece más de una vez?

## Parte 4 — Pipeline CDC con dp.create_auto_cdc_flow

In [ ]:
# ─────────────────────────────────────────────────────────
# Fuente Bronze: ingesta incremental con Auto Loader
# ─────────────────────────────────────────────────────────
@dp.table(
    comment="Bronze CDC: flujo incremental de eventos de cambio."
)
def bronze_cdc_mi_dataset():
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format",         "csv")
        .option("cloudFiles.schemaLocation", SCHEMA_LOCATION)
        .option("header",                    "true")
        .option("inferSchema",               "true")
        .load(RUTA_CDC)
        .withColumn("_ingest_ts", F.current_timestamp())
    )

In [ ]:
# ─────────────────────────────────────────────────────────
# SCD Tipo 1 (apply_as_deletes=True, track_history=False)
# Solo conserva el último estado de cada clave
# ─────────────────────────────────────────────────────────
dp.create_streaming_table("silver_scd1_mi_dataset")

dp.create_auto_cdc_flow(
    name             = "silver_scd1_mi_dataset",
    source           = "bronze_cdc_mi_dataset",
    keys             = [CLAVE_PK],
    sequence_by      = F.col(COL_SECUENCIA),
    apply_as_deletes = F.expr(f"{COL_OPERACION} = 'DELETE'"),
    except_column_list = ["operacion", "_ingest_ts"]
)

In [ ]:
# ─────────────────────────────────────────────────────────
# SCD Tipo 2 (track_history=True)
# Conserva historial completo — cada cambio como nueva fila
# ─────────────────────────────────────────────────────────
dp.create_streaming_table("silver_scd2_mi_dataset")

dp.create_auto_cdc_flow(
    name             = "silver_scd2_mi_dataset",
    source           = "bronze_cdc_mi_dataset",
    keys             = [CLAVE_PK],
    sequence_by      = F.col(COL_SECUENCIA),
    apply_as_deletes = F.expr(f"{COL_OPERACION} = 'DELETE'"),
    track_history    = True,
    except_column_list = ["operacion", "_ingest_ts"]
)

## Parte 5 — Verificar resultados CDC (notebook interactivo posterior)

In [ ]:
# Ejecutar DESPUÉS de correr la pipeline en Lakeflow
print("=== SCD1: último estado de cada entidad ===")
spark.table("workspace.default.silver_scd1_mi_dataset").show(truncate=False)

print("\n=== SCD2: historial completo de cambios ===")
spark.table("workspace.default.silver_scd2_mi_dataset") \
    .orderBy(CLAVE_PK, "__START_AT") \
    .show(truncate=False)

In [ ]:
# Comparar SCD1 vs SCD2
cnt_scd1 = spark.table("workspace.default.silver_scd1_mi_dataset").count()
cnt_scd2 = spark.table("workspace.default.silver_scd2_mi_dataset").count()
print(f"SCD1 (último estado):     {cnt_scd1:,} filas")
print(f"SCD2 (historial completo): {cnt_scd2:,} filas")
print(f"\nRastros de cambio = {cnt_scd2 - cnt_scd1:,} filas adicionales en SCD2")

In [ ]:
# Verificar que los DELETEs se manejan bien
# En SCD1 los registros eliminados deben haber desaparecido
# En SCD2 deben tener __END_AT no nulo
print("SCD2 — registros marcados como eliminados:")
spark.table("workspace.default.silver_scd2_mi_dataset") \
    .filter(F.col("__END_AT").isNotNull()) \
    .show(truncate=False)

## Parte 6 — Preguntas de negocio sobre el historial

1. ¿Cuántas versiones distintas tiene la entidad con más cambios en SCD2?
2. ¿Los registros eliminados en SCD1 sí desaparecieron? ¿Qué valor les asigna Lakeflow en SCD2?
3. Si una entidad se actualiza 3 veces, ¿cuántas filas tendrá en SCD2?
4. ¿Para qué caso de uso de tu dominio elegirías SCD2 sobre SCD1?

## Parte 7 — Reflexión final

1. ¿Cuándo es imprescindible usar SCD2 en lugar de SCD1?
2. ¿Qué representa la columna `__START_AT` en SCD2? ¿Y `__END_AT`?
3. ¿Cómo consultarías el estado de una entidad en una fecha específica (time-travel) usando SCD2?
4. ¿Qué desafíos presenta llevar CDC a producción cuando el volumen de cambios es muy alto?

---

## Entrega en Git

```bash
git add semana_05/laboratorios/lab_03_cdc_scd.ipynb
git commit -m "lab: semana05 lab03 CDC SCD1 SCD2 create_auto_cdc_flow <nombre-dataset> - <tu-nombre>"
git push origin feature/semana05-lakeflow-<tu-nombre>
```